In [ ]:
spark


# Connecting To ADLS Gen 2 Storage succesfull

In [ ]:
storage_account = "your_storage_account_name"
application_id = "your_application_id"
directory_id = "your_directory_id"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", "your_secret_value")
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{directory_id}/oauth2/token")


# Reading The Data

In [ ]:
# 0. Customers Dataset
customers_df=spark.read\
    .format("csv")\
    .option("header","true")\
     .option("inferSchema","true")\
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_customers_dataset.csv")

# 1. Geolocation Dataset
geolocation_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_geolocation_dataset.csv")

# 2. Order Items Dataset
items_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_order_items_dataset.csv")

# 3. Order Payments Dataset
payments_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_order_payments_dataset.csv")

# 4. Order Reviews Dataset
reviews_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_order_reviews_dataset.csv")

# 5. Orders Dataset
orders_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_orders_dataset.csv")

# 6. Products Dataset
products_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_products_dataset.csv")

# 7. Sellers Dataset
sellers_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_sellers_dataset.csv")




#Reading Data from PyMongo DB

In [ ]:
from pymongo import MongoClient

In [ ]:
# importing module
from pymongo import MongoClient

hostname = "your_host_name"
database = "your_database_name"
port = "your_port_number"
username = "your_username"
password = "your_password"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [ ]:
collection = mydatabase["product_categories"]

In [ ]:
import pandas as pd
mongo_data=pd.DataFrame(list(collection.find()))

#Data Cleaning

In [ ]:
from pyspark.sql.functions import *

In [ ]:
def cleaneData(df,name):
    print("cleaning"+name)
    return df.dropDuplicates().na.drop("all")
orders_df=cleaneData(orders_df,"orders")
orders_df.show(10)

In [ ]:
# Convert DateColumns
orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp")))\
    .withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date")))\
    .withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))



In [ ]:
# Calculating Delivery & Time delays
orders_df = orders_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("Delay Time", col("actual_delivery_time") - col("estimated_delivery_time"))


In [ ]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|actual_delivery_time|estimated_delivery_time|Delay Time|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+
|6f6665c1d76e55561...|12ae89712aa5a178c...|   delivered|              2017-02-23|2017-02-23 08:15:17|         2017-02-23 09:10:22|                   2017-03-02|                   2017-04-03|                   7|                     39|       -32|
|5c1f2bd7eff

# Joining DataSet

In [ ]:
order_customer_df=orders_df.join(customers_df,"customer_id","left")
order_payment_df=order_customer_df.join(payments_df,"order_id","left")
order_item_df=order_payment_df.join(items_df,"order_id","left")
order_item_product_df=order_item_df.join(products_df,"product_id","left")
final_df=order_item_product_df.join(sellers_df,"seller_id","left")


# Mongo Data Enrichment

In [ ]:
mongo_data.drop("_id",axis=1,inplace=True)
mongo_spark_df=spark.createDataFrame(mongo_data)
mongo_spark_df.show(5)

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
+---------------------+-----------------------------+
only showing top 5 rows


In [ ]:
final_df=final_df.join(mongo_spark_df,"product_category_name","left")

In [ ]:
final_df.show(20)

In [ ]:
display(final_df)

In [ ]:
final_df.write.mode("overwrite").parquet("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/silver")

In [ ]:
# IF YOU FACE ANY ERROR REGARDING DUPLICATES , YOU CAN RUN THIS CODE AND GET RID OF DUPLICATES AND YOU CAN SAVE IT IN SILVER LAYER
def remove_duplicates(df):
    columns=df.columns
    seen_column=set()
    column_to_remove=[]
    for column in columns:
        if column in seen_column:
            column_to_remove.append(column)
        else:
            seen_column.add(column)
    df_cleaned=df.drop(*column_to_remove)
    return df_cleaned
final_df=remove_duplicates(final_df)
